# Introduction

Stroke is the third leading cause of death, globally, according to [WHO 2021 data](https://www.who.int/news-room/fact-sheets/detail/the-top-10-causes-of-death), after Ischaemic Heart Disease and COVID-19, accounting for roughly 10% of total global deaths, and the rate has been increasing in the 2000-2019 and 2019-2021 periods.

This burden goes well beyond the death counts as stroke leads to major long term disability in survivors and places a heavy ongoing demand on healthcare systems everywhere with many patients needing extended rehabilitation and daily support that drives up costs fast in both personal hardship and economic terms.

Prediction modeling has therefore become a key focus area in recent years since spotting individuals at higher risk ahead of time can change some of those trajectories especially as targeted prevention steps, like blood pressure control or habit adjustments work much better when applied to the right people. In this repository the goal is to try to establish predictive models via machine learning algorithms, using Stroke Prediction Dataset to uncover connections between features in this dataset with stroke occurrence.

# Exploratory Data Analysis

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os

df = pd.read_csv(os.path.join('..', 'data', 'stroke-prediction-dataset.csv'))
df.sample(5)

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
3885,70693,Female,28.0,0,1,Yes,Private,Rural,111.27,19.1,smokes,0
2145,10716,Female,49.0,0,0,Yes,Private,Rural,107.46,32.1,never smoked,0
4375,33560,Female,81.0,0,1,Yes,Govt_job,Urban,90.11,28.6,never smoked,0
288,70970,Female,17.0,0,0,No,Self-employed,Urban,82.18,23.4,Unknown,0
3787,34965,Female,18.0,0,0,No,Private,Urban,95.87,23.0,never smoked,0


### Data Overview & Initial Cleaning

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [4]:
numeric_features = ['age', 'avg_glucose_level', 'bmi']
df[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
age,5110.0,43.226614,22.612647,0.08,25.000,45.000,61.00,82.00
avg_glucose_level,5110.0,106.147677,45.283560,55.12,77.245,91.885,114.09,271.74
bmi,4909.0,28.893237,7.854067,10.30,23.500,28.100,33.10,97.60


In [5]:
categorical_features = [
    'Residence_type', 'gender', 'hypertension', 'heart_disease', 'ever_married',
    'work_type', 'smoking_status'
]
for cf in categorical_features:
    print(f"{cf}: {df[cf].unique()}")

Residence_type: ['Urban' 'Rural']
gender: ['Male' 'Female' 'Other']
hypertension: [0 1]
heart_disease: [1 0]
ever_married: ['Yes' 'No']
work_type: ['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']
smoking_status: ['formerly smoked' 'never smoked' 'smokes' 'Unknown']


There are no sentinel or unexpected values than those listed in the dataset about file.

In [6]:
df['stroke'].value_counts()

stroke
0    4861
1     249
Name: count, dtype: int64

There is remarkable class imbalance in target variable and this will be potentially problematic for positive case (stroke+) predictions.

In [7]:
for col in df.columns:
    print(f'{col}: {df[col].isna().sum()} / {len(df)}')

id: 0 / 5110
gender: 0 / 5110
age: 0 / 5110
hypertension: 0 / 5110
heart_disease: 0 / 5110
ever_married: 0 / 5110
work_type: 0 / 5110
Residence_type: 0 / 5110
avg_glucose_level: 0 / 5110
bmi: 201 / 5110
smoking_status: 0 / 5110
stroke: 0 / 5110


Only bmi has missing values and there are 201/5110 incomplete rows. The most conservative approach is to discard rows that have missing data; i.e. Complete Case Analysis.

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
data = df[categorical_features + numeric_features + ['stroke']]
# data.drop(data[data['bmi'].isna()], inplace=True, axis=0)
# data.dropna(axis=0, subset='bmi', inplace=True)
data = data[~data['bmi'].isna()]
data.sample(5)

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke
4537,Urban,Female,0,0,Yes,Govt_job,smokes,51.0,81.38,34.1,0
2499,Urban,Male,0,0,Yes,Self-employed,Unknown,77.0,68.38,25.1,0
1209,Rural,Male,0,0,Yes,Private,Unknown,37.0,74.58,31.6,0
4310,Rural,Female,0,0,Yes,Govt_job,never smoked,45.0,68.66,25.3,0
4930,Urban,Male,0,0,No,children,Unknown,9.0,84.40,14.9,0


In [10]:
data.shape, data.isna().sum()

((4909, 11),
 Residence_type       0
 gender               0
 hypertension         0
 heart_disease        0
 ever_married         0
 work_type            0
 smoking_status       0
 age                  0
 avg_glucose_level    0
 bmi                  0
 stroke               0
 dtype: int64)